# DARTS — Differentiable Architecture Search (2019)

_Papers · Efficiency & Compression_

**Stop searching architectures by trial-and-error; make the choice of operation a continuous, gradient-trainable weight.**

---

This notebook is a practice scaffold for the **DARTS — Differentiable Architecture Search (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

DARTS turns architecture search into a continuous, gradient-trainable problem. On an edge we have several candidate operations; instead of picking one, we keep a **softmax-weighted mixture** of all of them, governed by architecture parameters $\alpha$. We build it in five steps: (1) sanity-check the softmax mixture, (2) make a toy task and candidate ops, (3) set up the bi-level optimizers, (4) run the first-order search, then (5) discretize and ablate.

### Step 1 — Sanity-check the softmax mixture

Each operation on an edge gets a learnable architecture parameter; the mixture weight is the **softmax** of those parameters (Eqn. 2). Before building anything, we reproduce the lesson's worked example to confirm softmax behaves as expected: a larger $\alpha$ earns a larger share of the mixture. Here `[2.0, 0.5, 0.5, 0.5]` should put about 60% of the weight on the first op.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Worked example: softmax over 4 architecture params.
alpha_we = torch.tensor([2.0, 0.5, 0.5, 0.5])
softmax_we = torch.softmax(alpha_we, dim=0)
print("worked example  softmax([2.0,0.5,0.5,0.5]) =",
      [round(v, 4) for v in softmax_we.tolist()])
# worked example  softmax([2.0,0.5,0.5,0.5]) = [0.599, 0.1337, 0.1337, 0.1337]

### Step 2 — Build a toy task and the candidate operations

The target `y` depends on **feature 0 only**, so a perfect search should learn to route through the op that reads feature 0. We split the data into a train and validation half — DARTS needs both, because $\alpha$ is scored on held-out data. Four candidate ops live on the edge: `op0` reads the useful feature, `op1`/`op2` read useless features, and `op3` is the **zero op** (outputs nothing).

In [ ]:
# Toy task: target y = feature 0 only. Train/val split.
N, ntr = 400, 300
g = torch.Generator().manual_seed(1)
X = torch.randn(N, 4, generator=g)
y = X[:, 0:1].clone()                      # depends ONLY on feature 0
Xtr, ytr = X[:ntr], y[:ntr]
Xva, yva = X[ntr:], y[ntr:]

# Four candidate operations on the edge. op_i has a learnable scalar weight w[i].
#   op0 = useful (uses feature 0); op1, op2 = useless features; op3 = zero op.
NAMES = ["pick-feat0 (USEFUL)", "pick-feat1", "pick-feat2", "zero"]

def op(idx, x, wi):
    if idx == 3:
        return torch.zeros(x.shape[0], 1)   # the zero op contributes nothing
    return wi * x[:, idx:idx+1]              # scale the chosen feature column

### Step 3 — Architecture params, weights, and the mixed operation

We hold two kinds of parameters. The architecture params $\alpha$ (one per op, all starting equal) decide the **mixture**; the ordinary weights `w` scale each op's output. They get **separate optimizers**: Adam updates $\alpha$ from validation loss, SGD updates `w` from training loss. The `mixed` function realizes Eqn. 2 — a softmax-weighted sum over all four ops.

In [ ]:
alpha = torch.zeros(4, requires_grad=True)         # the continuous architecture
w     = torch.ones(4,  requires_grad=True)         # the ordinary weights

opt_a = torch.optim.Adam([alpha], lr=0.05)         # alpha optimizer (validation)
opt_w = torch.optim.SGD([w], lr=0.1)               # weight optimizer (training)
lossfn = nn.MSELoss()

def mixed(Xb):                                     # Eqn. 2: softmax-weighted mixture of all ops
    wts = torch.softmax(alpha, dim=0)
    return sum(wts[i] * op(i, Xb, w[i]) for i in range(4))

### Step 4 — Run the bi-level first-order search

This is the heart of DARTS. Each iteration takes two gradient steps: first nudge $\alpha$ to lower the **validation** loss, then nudge `w` to lower the **training** loss. We use the cheap first-order approximation ($\xi = 0$): no second-order unrolling, just alternating updates. After 60 iterations $\alpha$ should concentrate on the useful op, and argmax gives the discrete choice.

In [ ]:
for ep in range(60):
    # alpha step on the VALIDATION split (xi = 0, first-order).
    opt_a.zero_grad()
    loss_val = lossfn(mixed(Xva), yva)
    loss_val.backward()
    opt_a.step()

    # weight step on the TRAINING split.
    opt_w.zero_grad()
    loss_tr = lossfn(mixed(Xtr), ytr)
    loss_tr.backward()
    opt_w.step()

probs = torch.softmax(alpha, dim=0).detach()
print("\nFinal mixture weights (softmax of alpha):")
for i in range(4):
    print(f"  {NAMES[i]:22s} alpha={alpha[i].item():+.3f}  softmax={probs[i].item():.4f}")

argmax = int(torch.argmax(alpha).item())
print("ARGMAX op (discrete choice) -> op%d: %s" % (argmax, NAMES[argmax]))
# Our small run: alpha concentrates on op0; softmax ~0.757 on the USEFUL op; argmax = op0.

### Step 5 — Discretize the child, then ablate

DARTS §2.4 says: keep only the argmax op and **retrain it from scratch** — this derived child is the actual architecture you ship. We retrain the selected op and check it generalizes (near-zero validation MSE). As an honest ablation, we retrain a *useless* op the same way: with no signal about the target, it should stay high. The contrast proves the search up-weighted the operation that carries the signal.

In [ ]:
# Discretize: keep the argmax op and RETRAIN it from scratch (the derived child).
wf = torch.ones(1, requires_grad=True)
opt_f = torch.optim.SGD([wf], lr=0.1)
for ep in range(60):
    opt_f.zero_grad()
    pred = wf * Xtr[:, argmax:argmax+1]
    lossfn(pred, ytr).backward()
    opt_f.step()
val_mse_good = lossfn(wf.detach() * Xva[:, argmax:argmax+1], yva).item()
print("derived (argmax) op retrained: w=%.4f  val MSE=%.4f" % (wf.item(), val_mse_good))
# derived (argmax) op retrained: w=1.0000  val MSE=0.0000   <- generalizes perfectly

# ABLATION: retrain a USELESS op the same way -> should NOT generalize.
wb = torch.ones(1, requires_grad=True)
opt_b = torch.optim.SGD([wb], lr=0.1)
for ep in range(60):
    opt_b.zero_grad()
    pred = wb * Xtr[:, 1:2]
    lossfn(pred, ytr).backward()
    opt_b.step()
val_mse_bad = lossfn(wb.detach() * Xva[:, 1:2], yva).item()
print("wrong op retrained:            w=%.4f  val MSE=%.4f" % (wb.item(), val_mse_bad))
# wrong op retrained:            w=-0.0440  val MSE=1.1024   <- no signal, cannot fit
# (Numbers are our small run, not the paper's reported result.)

## Visualize the data & results

_During a first-order DARTS search, does the softmax weight on the USEFUL operation rise above the useless ones, so argmax picks it?_

### Step 1 — Rebuild the toy search, recording the trajectory

To watch the architecture *evolve*, we re-run the same first-order search but log the softmax mixture after every iteration. The setup mirrors Step 2-3 above: the same target (feature 0 only), the same four candidate ops, and the same two optimizers.

In [ ]:
import torch
import torch.nn as nn

# Reproduce the qualitative DARTS effect on toy data: alpha concentrates on the right op.
N, ntr = 400, 300
g = torch.Generator().manual_seed(1)
X = torch.randn(N, 4, generator=g)
y = X[:, 0:1].clone()                       # target uses ONLY feature 0
Xtr, ytr = X[:ntr], y[:ntr]
Xva, yva = X[ntr:], y[ntr:]

def op(idx, x, wi):                          # candidate ops; op3 is the zero op
    if idx == 3:
        return torch.zeros(x.shape[0], 1)
    return wi * x[:, idx:idx+1]

torch.manual_seed(0)
alpha = torch.zeros(4, requires_grad=True)   # architecture params, all equal
w     = torch.ones(4,  requires_grad=True)
oa = torch.optim.Adam([alpha], lr=0.05)
ow = torch.optim.SGD([w], lr=0.1)
lf = nn.MSELoss()

def mixed(Xb):                               # Eqn. 2
    p = torch.softmax(alpha, dim=0)
    return sum(p[i] * op(i, Xb, w[i]) for i in range(4))

### Step 2 — Run the search and read the trajectory

We run the same alpha-on-val / w-on-train loop, but after each step we append the current mixture to `traj`. Printing every fifth row shows the useful op's share climbing (from a flat ~0.27 toward ~0.76) while the useless ops fade — and argmax lands on op0.

In [ ]:
traj = []
for ep in range(60):
    oa.zero_grad()
    lf(mixed(Xva), yva).backward()           # alpha on VAL (xi=0)
    oa.step()

    ow.zero_grad()
    lf(mixed(Xtr), ytr).backward()           # w on TRAIN
    ow.step()

    p = torch.softmax(alpha, dim=0).detach()
    traj.append([ep] + [round(v, 4) for v in p.tolist()])

for row in traj[::5]:
    print(row)            # [epoch, p_op0_USEFUL, p_op1, p_op2, p_op3]
print("argmax op:", int(torch.argmax(alpha).item()))
# op0 (useful) rises ~0.27 -> ~0.76; useless ops fall; argmax = op0.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your toy search has selected a single op by argmax over $\alpha$. To prove the
            search found the right op, you retrain the derived (argmax) op from scratch and, separately,
            retrain a useless op the same way. What do you expect each one's validation loss to be, and
            what does the contrast demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Take the argmax op, reset its weight, retrain it on the training split, and measure validation loss. — _This mirrors DARTS §2.4: derive the discrete child from $\alpha$, then retrain it from scratch._
- Pick one of the useless ops, retrain it identically, and measure its validation loss. — _An honest ablation changes exactly one thing &mdash; which op is kept &mdash; so any gap is attributable to the architecture choice._
- Compare: the derived op should reach near-zero validation loss; the useless op should stay high. — _If the search up-weighted the right op, only that op can fit unseen data &mdash; the others have no signal about the target._

**Answer:** The derived (argmax) op retrains to essentially perfect validation loss (in our run, MSE
                 $\approx 0.0000$ with its weight recovering $\approx 1.0$), while the useless op stays high
                 (in our run, MSE $\approx 1.10$ &mdash; no better than predicting near-zero). Since the two are
                 retrained identically and differ only in which op was kept, this isolates the architecture
                 choice: DARTS up-weighted the operation that actually carries the signal, and the discrete child
                 derived from $\alpha$ generalizes. The CODEVIZ panel shows $\alpha$ concentrating on that op
                 during the search.

</details>

**Problem 2.** On one edge the architecture parameters are $\alpha = [1.0, 1.0, 3.0]$ over three ops. Using Eqn. 2,
            what mixture weight does each op get, and which op would the final discretization keep?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Exponentiate: $e^{1.0} \approx 2.718$, $e^{1.0} \approx 2.718$, $e^{3.0} \approx 20.086$. — _Softmax first exponentiates each architecture parameter._
- Sum: $2.718 + 2.718 + 20.086 \approx 25.522$. — _The denominator of the softmax is the sum of the exponentials._
- Divide: $2.718/25.522 \approx 0.107$, $0.107$, and $20.086/25.522 \approx 0.787$. — _Each mixture weight is its exponential over the sum; the weights total $\approx 1$._

**Answer:** The mixture weights are about $[0.107,\,0.107,\,0.787]$. The third op dominates the blend
                 ($\approx 79\%$), and since it has the largest $\alpha$, argmax keeps op 3 at
                 discretization (§2.4). The other two would be dropped.

</details>

**Problem 3.** Why does DARTS train the architecture parameters $\alpha$ on a held-out validation split instead of
            on the training split, and what could go wrong if you trained both $\alpha$ and the weights $w$ on
            the same training data?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the bi-level objective (Eqns. 3-4): $w$ minimizes $\mathcal{L}_{\text{train}}$, $\alpha$ minimizes $\mathcal{L}_{\text{val}}$. — _The two roles are deliberately scored on different data: $\alpha$ is judged on data $w$ never trained on._
- Imagine collapsing both onto the training set: $\alpha$ would be rewarded for any op that lowers training loss, including by overfitting. — _Without held-out scoring, capacity that memorizes is indistinguishable from capacity that generalizes._
- Conclude that the validation split is what makes the search prefer generalizing architectures. — _An op only earns weight if it helps on unseen data &mdash; the same logic as a validation set in hyper-parameter tuning, with the architecture as the hyper-parameter._

**Answer:** $\alpha$ is the architecture &mdash; effectively a hyper-parameter &mdash; so it must be chosen on
                 data the weights did not fit, or the search would reward operations that merely overfit the
                 training set. Training both $\alpha$ and $w$ on the same data removes that safeguard: a
                 high-capacity op could be up-weighted for memorizing rather than generalizing. The held-out
                 validation split (the bi-level objective, Eqns. 3-4) is exactly what makes DARTS prefer
                 architectures that work on unseen data.

</details>